In [1]:
from typing import (
    Callable,
    Dict,
    Tuple,
)

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split

from src import dataset
from src.models import mk_model
from src.dataset import load_preprocessed_dataset
from src.configs import TrainingConfig, ModelConfig, DatasetConfig, DEVICE, PIXEL_VALUE_CHANNEL_IDX

In [2]:
dataset.mk_dataset(verbose=False)
train_cfg = TrainingConfig()
model_cfg = ModelConfig()
dataset_cfg = DatasetConfig()

x_train, y_train, x_test = load_preprocessed_dataset()
model = mk_model(model_cfg)
x_ds = torch.utils.data.TensorDataset(torch.cat((x_train, x_test)).to(DEVICE))
x_dl = DataLoader(x_ds, train_cfg.batch_size)
ssl_transform = v2.RandomAffine(degrees=(-10, 10), translate=(0.1, 0.3), scale=(0.75, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.starting_lr)

In [5]:
from random import randint


criterion_ret_t = Tuple[Tensor, Dict[str, Tensor]]
criterion_t = Callable[[Tensor, Tensor], criterion_ret_t]


def ssl_train_model(
        x_dl: DataLoader,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        mask_size: int,
        transform: callable,
    ):
    for (x, ) in x_dl:
        mask_slices = mk_mask_slices(mask_size)
        x, x_target = get_augmeted_x_and_target(x, mask_slices, transform)
        def criterion(x_pred: Tensor, x_target: Tensor) -> criterion_ret_t:
            x_target_pred = x_pred[:, PIXEL_VALUE_CHANNEL_IDX, mask_slices[0], mask_slices[1]]
            print("x:", x.shape)
            print("x_target:", x_target.shape)
            print("x_pred:", x_pred.shape)
            print("x_target_pred:", x_target_pred.shape)
            loss = torch.nn.functional.mse_loss(x_target_pred, x_target)
            return loss, {"mse_loss": loss}
        losses = model_step(model, x, x_target, optimizer, criterion)
        print(losses["mse_loss"])

# @torch.compile
def model_step(
        model: nn.Module,
        x: Tensor,
        y_true: Tensor,
        optimizer: torch.optim.Optimizer,
        criterion: criterion_t
    ) -> dict[str, Tensor]:
    optimizer.zero_grad()
    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
        y_pred_logits = model(x)
        loss, losses = criterion(y_pred_logits, y_true)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return losses

def get_augmeted_x_and_target(
        x: Tensor,
        mask_slices: tuple[slice, slice],
        transform: callable,
    ) -> tuple[Tensor, Tensor]:
    x = transform(x)
    x_target = x[:, 0, mask_slices[0], mask_slices[1]]
    x[:, 0, mask_slices[0], mask_slices[1]] = 0
    return x, x_target

def mk_mask_slices(mask_size: int) -> tuple[slice, slice]:
    h_start = randint(0, 256 - mask_size)
    w_start = randint(0, 256 - mask_size)
    w_slice = slice(h_start, h_start + mask_size)
    h_slice = slice(w_start, w_start + mask_size)
    return w_slice, h_slice

ssl_train_model(x_dl, model, optimizer, 30, ssl_transform)

x: torch.Size([128, 1, 256, 256])
x_target: torch.Size([128, 30, 30])
x_pred: torch.Size([128, 56, 256, 256])
x_target_pred: torch.Size([128, 30, 30])
tensor(0.0592, device='cuda:0', grad_fn=<MseLossBackward0>)
x: torch.Size([128, 1, 256, 256])
x_target: torch.Size([128, 30, 30])
x_pred: torch.Size([128, 56, 256, 256])
x_target_pred: torch.Size([128, 30, 30])
tensor(0.0271, device='cuda:0', grad_fn=<MseLossBackward0>)
x: torch.Size([128, 1, 256, 256])
x_target: torch.Size([128, 30, 30])
x_pred: torch.Size([128, 56, 256, 256])
x_target_pred: torch.Size([128, 30, 30])
tensor(0.0242, device='cuda:0', grad_fn=<MseLossBackward0>)
x: torch.Size([128, 1, 256, 256])
x_target: torch.Size([128, 30, 30])
x_pred: torch.Size([128, 56, 256, 256])
x_target_pred: torch.Size([128, 30, 30])
tensor(0.0259, device='cuda:0', grad_fn=<MseLossBackward0>)
x: torch.Size([128, 1, 256, 256])
x_target: torch.Size([128, 30, 30])
x_pred: torch.Size([128, 56, 256, 256])
x_target_pred: torch.Size([128, 30, 30])
tenso